# 03 — Aggregation and Agreement
Combines the LLM annotations into final labels by majority voting at two levels and measures agreement.

| Level | Voters | Purpose |
|---|---|---|
| 1. Within a model | the 3 runs of one model | self-consistency |
| 2. Across models | the 3 models | final label |

A label is accepted only if at least 2 of 3 voters agree. Only models whose own expected function equals the final expected function vote on Actual Function and Fulfillment; Mismatch Type is voted only by models that also agree with the final Fulfillment. Pairs need a final Expected, Actual and Fulfillment label to be kept.

Inputs: `llm_annotations_master.csv`, `dialogue_expectation_mismatch.csv`. Output: `final_annotated_dataset.csv`.

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from itertools import combinations
from sklearn.metrics import cohen_kappa_score

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## Settings

In [ ]:
MASTER_PATH = "llm_annotations_master.csv"
POOL_PATH   = "dialogue_expectation_mismatch.csv"

MODELS = ["Qwen3_14B", "Gemma3_12B", "Llama3_1_8B"]
RUNS   = [1, 2, 3]
MIN_VOTES = 2

STRICT_CONDITIONING = True
REQUIRED_DIMENSIONS = ["expected", "actual", "fulfillment"]   # Mismatch kept as an extra column

# column suffix used inside the master file for every label
STAGE2_SUFFIX = {"actual": "actual", "fulfillment": "fulfillment", "mismatch": "mismatch"}
LABEL_NAMES = {"expected": "Expected Response Function", "actual": "Actual Function",
               "fulfillment": "Fulfillment", "mismatch": "Mismatch Type"}

## Load annotations
A pair counts as annotated once all three models produced at least one Stage-1 output for it.

In [ ]:
raw = pd.read_csv(MASTER_PATH, encoding="utf-8-sig", dtype=str, keep_default_na=False)
# A pair counts as "annotated" once ALL three models have produced at least one Stage-1 output for it.
# (The raw output is stored even when it was invalid, so failed runs still count as attempted.)
done = pd.concat([raw[[f"{m}_s1_run{r}_raw" for r in RUNS]].ne("").any(axis=1) for m in MODELS], axis=1).all(axis=1)
master = raw[done].reset_index(drop=True)
N = len(master)
assert master["pair_id"].is_unique
print(f"Pairs in sample: {len(raw):,} | annotated: {N:,}")

## Voting and agreement functions
`vote()` returns the winning label and its status (Unanimous, Majority, No majority, Missing); a missing vote is never counted as a label. `krippendorff_alpha_nominal()` is Krippendorff's α for nominal data, checked against the reference example (α = 0.743).

In [ ]:
def vote(labels, min_votes=MIN_VOTES, n_voters=3):
    valid = [l for l in labels if isinstance(l, str) and l.strip() != ""]
    if len(valid) < min_votes:
        return None, "Missing"
    counts = Counter(valid)
    top, c = counts.most_common(1)[0]
    if c == n_voters:
        return top, "Unanimous"
    if c >= min_votes and list(counts.values()).count(c) == 1:
        return top, "Majority"
    return None, "No majority"


def krippendorff_alpha_nominal(data):
    """data: list of units; each unit is a list of values (one per coder, None = missing)."""
    coincidence = Counter()
    for unit in data:
        vals = [v for v in unit if v is not None]
        m = len(vals)
        if m < 2:
            continue
        for i, a in enumerate(vals):
            for j, b in enumerate(vals):
                if i != j:
                    coincidence[(a, b)] += 1.0 / (m - 1)
    n_c = Counter()
    for (a, b), w in coincidence.items():
        n_c[a] += w
    n = sum(n_c.values())
    if n <= 1:
        return np.nan
    d_o = sum(w for (a, b), w in coincidence.items() if a != b)
    d_e = sum(n_c[a] * n_c[b] for a in n_c for b in n_c if a != b) / (n - 1)
    return 1.0 - d_o / d_e if d_e else np.nan


# unit test: worked example from Krippendorff (2011) / Wikipedia -> alpha = 0.743
_ex = {"A": [1,2,3,3,2,1,4,1,2,None,None,None], "B": [1,2,3,3,2,2,4,1,2,5,None,3],
       "C": [None,3,3,3,2,3,4,2,2,5,1,None],    "D": [1,2,3,3,2,4,4,1,2,5,1,None]}
_units = [[_ex[k][i] for k in "ABCD"] for i in range(12)]
assert abs(krippendorff_alpha_nominal(_units) - 0.743) < 0.001
print("Krippendorff alpha implementation verified (0.743 on the reference example).")

## Level 1 — voting within each model

In [ ]:
level1 = {}   # (model, dim) -> list of (label, status)

for model in MODELS:
    level1[(model, "expected")] = [
        vote([master.at[i, f"{model}_s1_run{r}_label"] for r in RUNS]) for i in range(N)]
    for dim, suffix in STAGE2_SUFFIX.items():
        level1[(model, dim)] = [
            vote([master.at[i, f"{model}_s2_run{r}_{suffix}"] for r in RUNS]) for i in range(N)]

def l1_label(model, dim):  return pd.Series([x[0] for x in level1[(model, dim)]])
def l1_status(model, dim): return pd.Series([x[1] for x in level1[(model, dim)]])

rows = []
for dim in ["expected", "actual", "fulfillment", "mismatch"]:
    for model in MODELS:
        s = l1_status(model, dim).value_counts()
        rows.append({"Dimension": LABEL_NAMES[dim], "Model": model,
                     "Unanimous (3/3)": s.get("Unanimous", 0),
                     "Majority only (2/3)": s.get("Majority", 0),
                     "No majority": s.get("No majority", 0),
                     "Missing / not run": s.get("Missing", 0)})
within = pd.DataFrame(rows)
within["Agreed (≥2/3) %"] = ((within["Unanimous (3/3)"] + within["Majority only (2/3)"]) / N * 100).round(1)
within["Unanimous %"]     = (within["Unanimous (3/3)"] / N * 100).round(1)
print(f"TABLE A — Voting WITHIN each model (3 runs per model; percentages out of {N:,} pairs)")
display_cols = ["Dimension","Model","Unanimous (3/3)","Majority only (2/3)","No majority","Missing / not run","Agreed (≥2/3) %","Unanimous %"]
print(within[display_cols].to_string(index=False))

Krippendorff's α over the 3 runs of each model.

In [ ]:
rows = []
for dim, suffix in [("expected", None)] + list(STAGE2_SUFFIX.items()):
    for model in MODELS:
        if dim == "expected":
            cols = [f"{model}_s1_run{r}_label" for r in RUNS]
        else:
            cols = [f"{model}_s2_run{r}_{suffix}" for r in RUNS]
        units = [[(v if v != "" else None) for v in master.loc[i, cols]] for i in range(N)]
        rows.append({"Dimension": LABEL_NAMES[dim], "Model": model,
                     "Krippendorff α (3 runs)": round(krippendorff_alpha_nominal(units), 3)})
alpha_runs = pd.DataFrame(rows)
print("TABLE B — Run-to-run reliability inside each model")
print(alpha_runs.pivot(index="Dimension", columns="Model", values="Krippendorff α (3 runs)").loc[[LABEL_NAMES[d] for d in ["expected","actual","fulfillment","mismatch"]]].to_string())

## Level 2 — voting across models

In [ ]:
final = master[["pair_id"]].copy()
final["exp_votes"] = [[level1[(m, "expected")][i][0] for m in MODELS] for i in range(N)]
exp = [vote(v, n_voters=3) for v in final["exp_votes"]]
final["expected"], final["expected_agreement"] = [e[0] for e in exp], [e[1] for e in exp]

def across_models(strict):
    out = {}
    for dim in ["actual", "fulfillment"]:
        labels, status = [], []
        for i in range(N):
            votes = []
            for m in MODELS:
                eligible = (not strict) or (level1[(m, "expected")][i][0] == final.at[i, "expected"])
                votes.append(level1[(m, dim)][i][0] if eligible else None)
            lab, st = vote(votes)
            labels.append(lab); status.append(st)
        out[dim] = (labels, status)
    labels, status = [], []
    for i in range(N):
        ful = out["fulfillment"][0][i]
        if ful is None:
            labels.append(None); status.append("Missing"); continue
        votes = []
        for m in MODELS:
            eligible = (not strict) or (level1[(m, "expected")][i][0] == final.at[i, "expected"])
            same_ful = level1[(m, "fulfillment")][i][0] == ful
            votes.append(level1[(m, "mismatch")][i][0] if (eligible and same_ful) else None)
        lab, st = vote(votes)
        labels.append(lab); status.append(st)
    out["mismatch"] = (labels, status)
    return out

def build(strict):
    r = final[["pair_id", "expected", "expected_agreement"]].copy()
    res = across_models(strict)
    for dim in ["actual", "fulfillment", "mismatch"]:
        r[dim], r[dim + "_agreement"] = res[dim]
    return r

result = build(STRICT_CONDITIONING)
print("Final labels created (across-model majority):")
print({LABEL_NAMES[d]: int(result[d].notna().sum()) for d in LABEL_NAMES})

# consistency check: no impossible Fulfilment/Mismatch pair may survive
bad = ((result.fulfillment == "Fulfilled") & result.mismatch.notna() & (result.mismatch != "None")) | \
      ((result.fulfillment == "Not Fulfilled") & (result.mismatch == "None"))
assert bad.sum() == 0, "inconsistent Fulfilment/Mismatch combination found"
print("Consistency check passed: Fulfilled ↔ None, Not Fulfilled ↔ specific mismatch.")

### Agreement between models
Unanimity and Krippendorff's α use the models' level-1 labels for all pairs; pairwise Cohen's κ compares models on pairs where both share the expected function.

In [ ]:
rows = []
for dim in ["expected", "actual", "fulfillment", "mismatch"]:
    st = result[dim + "_agreement"].value_counts()
    # agreement between the three models' own (level-1) labels for ALL pairs, computed the same way for every dimension
    units = [[level1[(m, dim)][i][0] for m in MODELS] for i in range(N)]
    all_same = sum(1 for u in units if None not in u and len(set(u)) == 1)
    rows.append({"Dimension": LABEL_NAMES[dim],
                 "Unanimous (3/3 models)": all_same,
                 "Majority only (2/3)": st.get("Majority", 0),
                 "No majority": st.get("No majority", 0),
                 "Missing": st.get("Missing", 0),
                 "Final label created %": round(result[dim].notna().mean() * 100, 1),
                 "Krippendorff α (3 models)": round(krippendorff_alpha_nominal(units), 3)})
across = pd.DataFrame(rows)
print(f"TABLE C — Voting ACROSS models (strict conditioning = {STRICT_CONDITIONING}; out of {N:,} pairs)")
print(across.to_string(index=False))

rows = []
for dim in ["expected", "actual", "fulfillment", "mismatch"]:
    for a, b in combinations(MODELS, 2):
        la, lb = l1_label(a, dim), l1_label(b, dim)
        ok = la.notna() & lb.notna()
        if dim != "expected" and STRICT_CONDITIONING:  # compare only where both models share the expectation
            ea, eb = l1_label(a, "expected"), l1_label(b, "expected")
            ok &= (ea == eb)
        rows.append({"Dimension": LABEL_NAMES[dim], "Model pair": f"{a} vs {b}", "Pairs compared": int(ok.sum()),
                     "Cohen κ": round(cohen_kappa_score(la[ok], lb[ok]), 3),
                     "Raw agreement %": round((la[ok] == lb[ok]).mean() * 100, 1)})
kappa = pd.DataFrame(rows)
print("\nTABLE D — Pairwise agreement between models")
print(kappa.to_string(index=False))

## Funnel to the analysis dataset

In [ ]:
need = [d for d in REQUIRED_DIMENSIONS]
keep = result[need].notna().all(axis=1)

funnel, mask = [("Pairs annotated by all three models", N)], pd.Series(True, index=result.index)
for d in need:
    mask &= result[d].notna()
    funnel.append((f"… and final {LABEL_NAMES[d]} agreed (≥2 models)", int(mask.sum())))
funnel.append(("Final analysis dataset", int(keep.sum())))
funnel_df = pd.DataFrame(funnel, columns=["Step", "Pairs remaining"])
funnel_df["% of annotated"] = (funnel_df["Pairs remaining"] / N * 100).round(1)
print("TABLE E — Funnel from annotated pairs to the analysis dataset")
print(funnel_df.to_string(index=False))

### Sensitivity of the voting rule

In [ ]:
def usable(strict, dims):
    r = build(strict)
    return int(r[dims].notna().all(axis=1).sum())

three, four = ["expected", "actual", "fulfillment"], ["expected", "actual", "fulfillment", "mismatch"]
sens = pd.DataFrame([
    {"Voting variant": "Models vote on Stage-2 labels regardless of their own expectation", "Required labels": "3 analysed", "Usable pairs": usable(False, three)},
    {"Voting variant": "Models vote on Stage-2 labels regardless of their own expectation", "Required labels": "all 4",       "Usable pairs": usable(False, four)},
    {"Voting variant": "Only models that share the final expectation vote  (CHOSEN)",       "Required labels": "3 analysed", "Usable pairs": usable(True, three)},
    {"Voting variant": "Only models that share the final expectation vote",                  "Required labels": "all 4",       "Usable pairs": usable(True, four)},
])
print(sens.to_string(index=False))

## Save the final dataset

In [ ]:
pool = pd.read_csv(POOL_PATH, encoding="utf-8-sig", dtype=str, keep_default_na=False)
base = pool[["pair_id", "dialogue_id", "turn", "context", "previous_utterance", "response",
             "previous_act_name", "response_act_name", "act_relation"]]
out = base.merge(result[["pair_id", "expected", "expected_agreement", "actual", "actual_agreement",
                         "fulfillment", "fulfillment_agreement", "mismatch", "mismatch_agreement"]], on="pair_id")
out = out[out["pair_id"].isin(result.loc[keep, "pair_id"])]
out = out.rename(columns={"expected": "Expected Response Function", "actual": "Actual Function",
                          "fulfillment": "Fulfillment", "mismatch": "Mismatch Type"})
out.to_csv("final_annotated_dataset.csv", index=False, encoding="utf-8-sig")
print(f"Saved final_annotated_dataset.csv  ({len(out):,} pairs, {out['dialogue_id'].nunique():,} dialogues)")

## Label distributions

In [ ]:
for col in ["Expected Response Function", "Actual Function", "Fulfillment"]:
    vc = out[col].value_counts()
    print(f"\n{col}\n" + pd.DataFrame({"n": vc, "%": (vc / len(out) * 100).round(1)}).to_string())

ful = (out.groupby("Expected Response Function")["Fulfillment"]
          .apply(lambda s: (s == "Fulfilled").mean() * 100).round(1).rename("Fulfilled %"))
print("\nFulfilment rate by expected function")
print(pd.concat([out["Expected Response Function"].value_counts().rename("n"), ful], axis=1).to_string())